# Read Alignment with BWA-MEM

**BioNexus Hub - Track 2: Genomics & NGS**

Map sequencing reads to a reference genome with BWA-MEM, then inspect the SAM/BAM result with samtools. Everything runs here in Colab - nothing to install on your computer.

Run each cell in order with Shift+Enter.


## 1. Install the tools

`bwa` is the aligner; `samtools` lets us convert and inspect the result.


In [ ]:
!apt-get -qq install -y bwa samtools
!echo '--- bwa ---'; bwa 2>&1 | head -3
!echo '--- samtools ---'; samtools --version | head -1

## 2. Make a tiny reference and some reads

Real references are huge, so for a fast demo we build a small 2,000 bp reference and sample 200 reads from it (with a touch of sequencing error). Because the reads come from this reference, they should map back to it cleanly.


In [ ]:
import random
random.seed(42)
bases = 'ACGT'

# A small reference 'chromosome'
ref = ''.join(random.choice(bases) for _ in range(2000))
with open('reference.fasta','w') as f:
    f.write('>chr1 demo reference\n')
    for i in range(0, len(ref), 60):
        f.write(ref[i:i+60] + '\n')

# 200 reads of length 100, with ~1% sequencing errors
L = 100
with open('reads.trimmed.fastq','w') as f:
    for n in range(200):
        pos = random.randint(0, len(ref)-L)
        seq = list(ref[pos:pos+L])
        for j in range(L):
            if random.random() < 0.01:
                seq[j] = random.choice(bases)
        seq = ''.join(seq)
        f.write(f'@read{n}_pos{pos+1}\n{seq}\n+\n{"I"*L}\n')

print('Reference written: 2000 bp')
print('Reads written: 200')

In [ ]:
# Peek at the files
!head -2 reference.fasta
!echo '...'; head -4 reads.trimmed.fastq

## 3. Index the reference

BWA builds a lookup index so it can find matches fast. You do this once per reference.


In [ ]:
!bwa index reference.fasta
!ls reference.fasta*

## 4. Align the reads

`bwa mem` produces alignments in SAM (text) format. We save a SAM version to read by eye, and a compact BAM version (via samtools) for everything downstream.


In [ ]:
# Text SAM (easy to peek at)
!bwa mem reference.fasta reads.trimmed.fastq > aligned.sam 2> bwa.log
# Compact BAM (what real pipelines keep)
!bwa mem reference.fasta reads.trimmed.fastq 2>> bwa.log | samtools view -b -o aligned.bam
print('Done. aligned.sam and aligned.bam created.')

## 5. Read the SAM file

Lines starting with `@` are the header. Each alignment line that follows is one read, with columns: QNAME, FLAG, RNAME (chromosome), POS (position), MAPQ (mapping quality), CIGAR, and so on. Notice the read names carry the true position we generated them from - compare it to the POS column.


In [ ]:
# Header lines
!grep '^@' aligned.sam | head
print('--- first few alignments (columns 1-6) ---')
!grep -v '^@' aligned.sam | head -5 | cut -f1-6

## 6. Sanity check: did the reads map?

`samtools flagstat` summarizes the alignment. Because our reads came straight from the reference, expect a very high mapped percentage. In a real project, a low rate means the wrong reference or contaminated reads.


In [ ]:
!samtools flagstat aligned.bam

## You did it

You turned raw reads into mapped reads and read the SAM/BAM format that every downstream genomics tool depends on.

**Try next:** change `random.seed(42)` to another number, or lower the read length `L`, and watch how the mapping rate and MAPQ values respond.

Next lesson: the **samtools survival kit** - sort and index this BAM so tools can jump to any region instantly.
